# Getting Started with Reinforcement Learning

This notebook covers:
1. **Basics**: Running a standard Gymnasium environment.
2. **Training**: Using Stable Baselines3 to train an agent.
3. **Custom Environments**: A template for creating your own RL world.

## 1. Running a Standard Environment
Let's test the installation by running the `CartPole-v1` environment with random actions.

In [ ]:
import gymnasium as gym

# Create the environment
env = gym.make("CartPole-v1", render_mode="human")

observation, info = env.reset()

for _ in range(100):
    action = env.action_space.sample()  # Random action
    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()

env.close()
print("Environment ran successfully!")

## 2. Training an Agent
We'll use Proximal Policy Optimization (PPO) from Stable Baselines3.

In [ ]:
from stable_baselines3 import PPO
import gymnasium as gym

env = gym.make("CartPole-v1")

# Initialize the agent
model = PPO("MlpPolicy", env, verbose=1)

# Train the agent
model.learn(total_timesteps=10000)

# Save the model
model.save("ppo_cartpole")
print("Model trained and saved.")

## 3. Custom Environment Template
If you want to build your own environment, use this template.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class MyCustomEnv(gym.Env):
    def __init__(self):
        super(MyCustomEnv, self).__init__()
        # Define action and observation space
        # They must be gymnasium.spaces objects
        # Example: 3 discrete actions (0, 1, 2)
        self.action_space = spaces.Discrete(3)
        # Example: Observation is a 1D array of 4 floats
        self.observation_space = spaces.Box(low=-1, high=1, shape=(4,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Initialize state
        state = np.zeros((4,), dtype=np.float32)
        return state, {}

    def step(self, action):
        # Apply action, calculate next_state, reward, done
        next_state = np.random.uniform(-1, 1, size=(4,)).astype(np.float32)
        reward = 1.0
        terminated = False
        truncated = False
        info = {}

        return next_state, reward, terminated, truncated, info

    def render(self):
        pass

print("Custom environment template defined.")

## Signal Analytics Dashboard and Tuning Workflow

The Streamlit dashboard (`src/analytics_dashboard.py`) evaluates how well the agent's actions match future price movement.

### How the dashboard works
- Loads a CSV dataset (`data/tech_training_data.csv` or `data/mock_data.csv`) and PPO model (`models/ppo_trading_bot` or `.zip`).
- Replays the model in `TradingEnv` and logs step-by-step actions (`Hold`, `Buy`, `Sell`).
- Computes forward return for a configurable horizon (`horizon_steps`).
- Converts forward return into a **true signal** using a movement threshold:
  - return > threshold => Buy
  - return < -threshold => Sell
  - otherwise => Hold
- Compares predicted action vs true signal and reports:
  - overall and actionable accuracy
  - buy/sell precision and recall
  - confusion matrix
  - trade win rate and cumulative signal return
  - action distribution and detailed signal log

### Tuning approach
1. Sweep `threshold` and `horizon_steps` to find a stable precision/recall balance (not only max accuracy).
2. Prefer settings with robust buy/sell quality and sensible action frequency (avoid overtrading).
3. Retrain PPO and re-check dashboard metrics after each training change:
   - reward shaping
   - training timesteps
   - penalties for churn/transaction costs
4. Validate on out-of-sample periods (time-based split) before trusting improvements.
5. Track metric trends over runs to confirm gains are consistent, not one-off.



## Sentiment Integration into Training and Environment

### Objective
Incorporate daily news sentiment into the RL state so action selection can condition on both price behavior and recent news tone.

### Integration path
1. `src/news_data.py` fetches Yahoo ticker news and computes article-level sentiment from title/summary.
2. News is aggregated to daily ticker features (`NewsCount`, `SentimentMean`, `SentimentStd`, `SentimentMin`, `SentimentMax`).
3. `src/market_data.py` merges aggregated sentiment into the date-level training basket via `merge_news_features(...)`.
4. Missing sentiment values are filled with `0.0` (neutral defaults).
5. `TradingEnv` includes sentiment columns automatically when present in the dataframe.

### Resulting observation layout
- Market: `Open`, `High`, `Low`, `Close`, `Volume`
- Sentiment (optional): `NewsCount`, `SentimentMean`, `SentimentStd`, `SentimentMin`, `SentimentMax`
- Portfolio: `balance`, `shares_held`

### Practical implications
- Old datasets without sentiment still run (backward compatible).
- Enriched datasets increase observation dimensionality and can change policy behavior.
- Retraining is required to exploit sentiment signals.

### Validation checklist
- Confirm merged training columns include sentiment fields.
- Run `tests/test_script.py` after integration changes.
- Compare dashboard precision/recall and confusion matrix before vs after sentiment-enabled retraining.

